# Gradient Gauge Fraction in a ReLU MLP
## Companion notebook for `run_experiment.py`

This notebook is the analysis companion for
`experiments/Tier_1_Core_Mechanism_Tests/KILL_EXPERIMENT_gauge_fraction_relu/run_experiment.py`.
It imports that script's `run_experiment()` function and works from its returned
results rather than re-implementing the experiment core.


## Scope and toy status

- This is a **toy mechanistic diagnostic**, not a broad falsification of Muon.
- The setup is deliberately narrow: full-batch NumPy training on random regression,
  width 32, depth 6, and only 3 seeds.
- The **primary analysis** is the set of hidden layers (all layers except the final
  linear output layer).
- The **last layer is treated as a control**, because it has no ReLU after it and is
  therefore less informative about nonlinear hidden-layer behavior.


## Measured vs. not measured

**Measured here**

For each layer and training step, the script computes

\[
	ext{gauge\_fraction} = rac{\lVert Q\,\mathrm{sym}(Q^T G)Vert_F^2}{\lVert GVert_F^2}, \qquad Q = \mathrm{polar}(W).
\]

This is the **Stiefel-normal fraction of the raw gradient** relative to the polar
factor of the current weight matrix.

**Not directly established here**

- That this quantity is exactly the part of the gradient that Muon removes.
- That Muon's practical update is identical to the tangent projection `G_tangent`.
- Any broad conclusion outside this one toy random-regression setup.
- Any formal statistical claim; with `n=3` seeds, the summaries here are descriptive.


In [ ]:
from pathlib import Path
import importlib.util
import sys
import textwrap

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt

try:
    from IPython.display import display
except Exception:
    def display(obj):
        print(obj)

try:
    plt.style.use('seaborn-v0_8-whitegrid')
except OSError:
    pass

def locate_experiment_script():
    relative = Path('experiments/Tier_1_Core_Mechanism_Tests/KILL_EXPERIMENT_gauge_fraction_relu/run_experiment.py')
    search_roots = [Path.cwd(), *Path.cwd().parents]

    for root in search_roots:
        candidate = (root / relative).resolve()
        if candidate.exists():
            return candidate

    for root in search_roots:
        candidate = (root / 'run_experiment.py').resolve()
        if candidate.exists() and candidate.parent.name == 'KILL_EXPERIMENT_gauge_fraction_relu':
            return candidate

    raise FileNotFoundError('Could not locate run_experiment.py from the current working directory.')

SCRIPT_PATH = locate_experiment_script()

def load_experiment_module(script_path):
    module_name = 'gauge_fraction_relu_experiment'
    if module_name in sys.modules:
        del sys.modules[module_name]
    spec = importlib.util.spec_from_file_location(module_name, script_path)
    module = importlib.util.module_from_spec(spec)
    sys.modules[module_name] = module
    spec.loader.exec_module(module)
    return module

experiment = load_experiment_module(SCRIPT_PATH)
SCRIPT_PATH


## Reproducibility, runtime, and configuration

The next cell runs the exact experiment implementation from the script and logs the
configuration, seeds, and basic package versions used for this notebook run.


In [ ]:
results = experiment.run_experiment(verbose=False)

runtime_rows = [
    {'item': 'script_path', 'value': str(SCRIPT_PATH)},
    {'item': 'runtime_seconds', 'value': f"{results['runtime_seconds']:.2f}"},
    {'item': 'numpy_version', 'value': np.__version__},
    {'item': 'matplotlib_version', 'value': matplotlib.__version__},
    {'item': 'pandas_version', 'value': pd.__version__},
]
config_rows = [{'parameter': key, 'value': value} for key, value in results['config'].items()]

print('Seeds used:', results['seeds'])
display(pd.DataFrame(runtime_rows))
display(pd.DataFrame(config_rows))


## High-level descriptive summary

The first table reports mean ± seed-level standard deviation for the main gauge-fraction
summaries. The primary column is the hidden-layer summary; the last-layer column is a
control. The all-layer column is retained for continuity with the original script.


In [ ]:
def mean_sd_text(mean_value, sd_value):
    return f'{mean_value:.2f} ± {sd_value:.2f}'

overall_rows = []
for condition_name in results['condition_order']:
    summary = results['summaries'][condition_name]
    overall_rows.append({
        'condition': results['condition_labels'][condition_name],
        'all layers (%)': mean_sd_text(summary['all_layers_mean_pct'], summary['all_layers_sd_pct']),
        'hidden layers (primary, %)': mean_sd_text(summary['hidden_layers_mean_pct'], summary['hidden_layers_sd_pct']),
        'last layer (control, %)': mean_sd_text(summary['last_layer_mean_pct'], summary['last_layer_sd_pct']),
        'final loss': mean_sd_text(summary['final_loss_mean'], summary['final_loss_sd']),
    })

overall_summary_df = pd.DataFrame(overall_rows)
display(overall_summary_df)

seed_rows = {'seed': results['seeds']}
for condition_name in results['condition_order']:
    seed_rows[f"{results['condition_labels'][condition_name]} hidden (%)"] = np.round(
        results['summaries'][condition_name]['per_seed_hidden_layers_pct'], 3
    )
seed_summary_df = pd.DataFrame(seed_rows)
display(seed_summary_df)


### Interpretation of the headline table

- **Primary question:** are the hidden-layer gauge fractions in the ReLU model clearly non-zero?
- **Control question:** does the final linear layer behave differently from the hidden layers?
- Because `n=3`, treat `mean ± SD` as descriptive uncertainty only.


In [ ]:
relu_hidden = results['summaries']['relu_sgd']['hidden_layers_mean_pct']
linear_hidden = results['summaries']['linear_sgd']['hidden_layers_mean_pct']
relu_last = results['summaries']['relu_sgd']['last_layer_mean_pct']
linear_last = results['summaries']['linear_sgd']['last_layer_mean_pct']

print(f'Hidden-layer headline: ReLU + SGD = {relu_hidden:.2f}% | Linear + SGD = {linear_hidden:.2f}%')
print(f'Last-layer control:   ReLU + SGD = {relu_last:.2f}% | Linear + SGD = {linear_last:.2f}%')
print('These are descriptive summaries of the raw-gradient normal fraction, not a direct measurement of the exact Muon-removed component.')


## Gauge fraction over time

The left panel shows the **hidden-layer primary analysis**. The right panel shows the
**last-layer control**. Shaded bands are seed-level standard deviations around the mean.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5), sharex=True)
steps = np.arange(results['config']['num_steps'])

for condition_name in results['condition_order']:
    summary = results['summaries'][condition_name]
    color = results['condition_colors'][condition_name]
    label = results['condition_labels'][condition_name]

    hidden_mean = summary['per_step_mean_hidden_layers_pct']
    hidden_sd = summary['per_step_sd_hidden_layers_pct']
    axes[0].plot(steps, hidden_mean, label=label, color=color, linewidth=2)
    axes[0].fill_between(steps, hidden_mean - hidden_sd, hidden_mean + hidden_sd, color=color, alpha=0.18)

    last_mean = summary['per_step_mean_last_layer_pct']
    last_sd = summary['per_step_sd_last_layer_pct']
    axes[1].plot(steps, last_mean, label=label, color=color, linewidth=2)
    axes[1].fill_between(steps, last_mean - last_sd, last_mean + last_sd, color=color, alpha=0.18)

axes[0].set_title('Hidden-layer gauge fraction over time (primary)')
axes[1].set_title('Last-layer gauge fraction over time (control)')
for ax in axes:
    ax.set_xlabel('Training step')
    ax.set_ylabel('Gauge fraction (%)')
    ax.legend(frameon=True)

plt.tight_layout()
plt.show()


### Time-course interpretation

The next table compresses the curves into early-vs-late summaries. This helps separate
a transient initialization effect from a signal that remains visible later in training.


In [ ]:
time_rows = []
for condition_name in results['condition_order']:
    summary = results['summaries'][condition_name]
    windows = summary['time_windows_pct']
    time_rows.append({
        'condition': results['condition_labels'][condition_name],
        'hidden early (%)': round(windows['early_hidden_layers_pct'], 3),
        'hidden late (%)': round(windows['late_hidden_layers_pct'], 3),
        'last early (%)': round(windows['early_last_layer_pct'], 3),
        'last late (%)': round(windows['late_last_layer_pct'], 3),
    })

display(pd.DataFrame(time_rows))


## Per-layer summary

The figure below reports the mean gauge fraction for each layer, averaged over all steps
and seeds. The dashed divider marks the transition to the final output layer, which we
interpret as a control rather than the primary nonlinear hidden-layer target.


In [ ]:
layer_numbers = np.arange(1, results['config']['num_layers'] + 1)

fig, ax = plt.subplots(figsize=(10, 5))
for condition_name in results['condition_order']:
    per_layer = results['summaries'][condition_name]['per_layer_mean_pct']
    ax.plot(
        layer_numbers,
        per_layer,
        marker='o',
        linewidth=2,
        label=results['condition_labels'][condition_name],
        color=results['condition_colors'][condition_name],
    )

ax.axvline(results['config']['num_layers'] - 0.5, linestyle='--', color='black', alpha=0.4)
ax.text(results['config']['num_layers'] - 0.45, ax.get_ylim()[1] * 0.95, 'last-layer control', va='top')
ax.set_xticks(layer_numbers)
ax.set_xlabel('Layer index')
ax.set_ylabel('Mean gauge fraction (%)')
ax.set_title('Per-layer mean gauge fraction by condition')
ax.legend(frameon=True)
plt.tight_layout()
plt.show()

per_layer_df = pd.DataFrame(
    {
        results['condition_labels'][condition_name]: np.round(
            results['summaries'][condition_name]['per_layer_mean_pct'], 3
        )
        for condition_name in results['condition_order']
    },
    index=[f'Layer {idx}' for idx in layer_numbers],
)
display(per_layer_df)


### Per-layer interpretation

For this first pass, the key question is not whether every layer exactly matches a single
theoretical number, but whether the hidden-layer signal is consistently non-negligible and
how it compares with the linear reference and the last-layer control.


## Loss trajectories

The experiment also tracks training loss for each of the four conditions. This is useful
for checking that the gauge-fraction summaries are being read along sensible training
trajectories rather than in a clearly broken regime.


In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
steps = np.arange(results['config']['num_steps'])

for condition_name in results['condition_order']:
    summary = results['summaries'][condition_name]
    mean_loss = summary['loss_per_step_mean']
    sd_loss = summary['loss_per_step_sd']
    color = results['condition_colors'][condition_name]
    label = results['condition_labels'][condition_name]
    ax.plot(steps, mean_loss, label=label, color=color, linewidth=2)
    ax.fill_between(steps, mean_loss - sd_loss, mean_loss + sd_loss, color=color, alpha=0.18)

ax.set_xlabel('Training step')
ax.set_ylabel('Loss')
ax.set_title('Loss over time by condition')
ax.legend(frameon=True)
plt.tight_layout()
plt.show()

loss_summary_df = pd.DataFrame([
    {
        'condition': results['condition_labels'][condition_name],
        'initial loss': mean_sd_text(
            results['summaries'][condition_name]['initial_loss_mean'],
            results['summaries'][condition_name]['initial_loss_sd'],
        ),
        'final loss': mean_sd_text(
            results['summaries'][condition_name]['final_loss_mean'],
            results['summaries'][condition_name]['final_loss_sd'],
        ),
    }
    for condition_name in results['condition_order']
])
display(loss_summary_df)


### Loss-trajectory interpretation

The loss curves matter for interpretation because they show the gauge-fraction diagnostic
along actual optimization trajectories for the four conditions. They do **not** by themselves
establish why one optimizer might work better than another.


## Legacy T1–T6 rule-based checks

These checks are preserved from the original script for continuity, but they should be read
as **rule-based descriptive checks**, not as formal hypothesis tests. They are reported on the
**legacy all-layer summary** unless otherwise noted.


In [ ]:
def format_rule_metric(rule_name, rule):
    if rule_name == 'T4':
        return f"ReLU {rule['metric_value_pct']:.2f}% vs Linear {rule['reference_value_pct']:.2f}%"
    if rule_name == 'T5':
        values = ', '.join(f'{value:.2f}%' for value in rule['metric_values_pct'])
        return f'[{values}] | count={rule["count_above_threshold"]}'
    return f"{rule['metric_value_pct']:.2f}%"

rule_rows = []
for rule_name in ['T1', 'T2', 'T3', 'T4', 'T5', 'T6']:
    rule = results['legacy_rule_checks'][rule_name]
    rule_rows.append({
        'rule': rule_name,
        'pass': rule['pass'],
        'scope': rule['scope'],
        'description': rule['description'],
        'target': rule['target'],
        'metric value(s)': format_rule_metric(rule_name, rule),
    })

rule_df = pd.DataFrame(rule_rows)
display(rule_df)
print(f"Legacy rule checks passed: {results['legacy_rule_pass_count']}/{results['legacy_rule_total']}")


## Calibrated conclusion

The conclusion below is intentionally narrower than the original notebook narrative. It is
phrased to match what the implemented metric actually shows in this toy setup.


In [ ]:
relu_sgd = results['summaries']['relu_sgd']
relu_muon = results['summaries']['relu_muon']
linear_sgd = results['summaries']['linear_sgd']
linear_muon = results['summaries']['linear_muon']

conclusion = f"""
Headline numbers from this run:
  - ReLU + SGD all-layer gauge fraction: {relu_sgd['all_layers_mean_pct']:.2f}%
  - ReLU + Muon all-layer gauge fraction: {relu_muon['all_layers_mean_pct']:.2f}%
  - Linear + SGD all-layer gauge fraction: {linear_sgd['all_layers_mean_pct']:.2f}%
  - Linear + Muon all-layer gauge fraction: {linear_muon['all_layers_mean_pct']:.2f}%

  - ReLU + SGD hidden-layer gauge fraction (primary): {relu_sgd['hidden_layers_mean_pct']:.2f}%
  - ReLU + SGD last-layer control: {relu_sgd['last_layer_mean_pct']:.2f}%

Calibrated reading:
  In this width-32, depth-6, random-regression toy experiment, the hidden-layer
  Stiefel-normal fraction of the raw gradient is clearly non-zero and remains visible
  throughout training. The ReLU model also sits below the linear reference in the
  legacy all-layer summary.

  What this supports:
  - a non-negligible gauge-fraction-style signal remains present in this toy nonlinear setup;
  - the hidden layers are consistent with the idea that ReLU does not erase the signal.

  What this does not support on its own:
  - that the measured gauge fraction is exactly the component Muon removes;
  - that Muon's full mechanism has been established;
  - that the conclusion automatically generalizes beyond this specific toy experiment.
"""

print(textwrap.dedent(conclusion).strip())
